# Unit IV - Relationships between Words
## Practical Implementation for BBA Students
---

In [ ]:
!pip install nltk pandas matplotlib seaborn networkx scikit-learn scipy

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

## 1. Tokenizing by N-grams

In [ ]:
from nltk import word_tokenize, ngrams
from collections import Counter

text = """The phone has amazing camera quality and the battery life is excellent.
I love the screen display but the battery drains too fast.
The camera takes great photos. Battery charges quickly though.
Screen resolution is high and camera zoom is the best feature."""

# Tokenize
tokens = word_tokenize(text.lower())
tokens = [t for t in tokens if t.isalpha()]  # Keep only words

# Generate different N-grams
print("=== N-GRAM TOKENIZATION ===")
for n in [1, 2, 3]:
    n_grams = list(ngrams(tokens, n))
    names = {1: 'Unigrams', 2: 'Bigrams', 3: 'Trigrams'}
    print(f"\n{names[n]} (first 8):")
    for gram in n_grams[:8]:
        print(f"  {' '.join(gram)}")

## 2. Counting and Filtering N-grams

In [ ]:
from nltk.corpus import stopwords
import pandas as pd
import matplotlib.pyplot as plt

stop_words = set(stopwords.words('english'))

# Generate bigrams
bigrams_all = list(ngrams(tokens, 2))

# Count ALL bigrams
bigram_counts_all = Counter([' '.join(b) for b in bigrams_all])

# Filter: Remove bigrams where EITHER word is a stopword
bigrams_filtered = [b for b in bigrams_all 
                    if b[0] not in stop_words and b[1] not in stop_words]
bigram_counts_filtered = Counter([' '.join(b) for b in bigrams_filtered])

print("=== BIGRAM COUNTING ===")
print(f"Total bigrams: {len(bigrams_all)}")
print(f"After removing stopwords: {len(bigrams_filtered)}")

print("\nTop 10 Bigrams (BEFORE filtering):")
for bg, count in bigram_counts_all.most_common(10):
    has_stop = any(w in stop_words for w in bg.split())
    flag = " ← stopword" if has_stop else " ✓ meaningful"
    print(f"  {bg:<25} {count}{flag}")

print("\nTop 10 Bigrams (AFTER filtering):")
for bg, count in bigram_counts_filtered.most_common(10):
    print(f"  {bg:<25} {count}")

# Visualize top filtered bigrams
top_bg = bigram_counts_filtered.most_common(10)
if top_bg:
    df_bg = pd.DataFrame(top_bg, columns=['Bigram', 'Count'])
    plt.figure(figsize=(10, 5))
    plt.barh(df_bg['Bigram'], df_bg['Count'], color='steelblue')
    plt.xlabel('Count', fontsize=12)
    plt.title('Top Filtered Bigrams (Stopwords Removed)', fontsize=14, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('top_bigrams.png', dpi=100)
    plt.show()

## 3. Bigrams for Sentiment Analysis Context

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

# Reviews with negation
reviews = [
    "This product is not good at all.",
    "The service was not bad actually.",
    "I am never disappointed with this brand.",
    "The quality is not what I expected.",
    "I don't like the design.",
    "The battery doesn't last long.",
    "This phone never fails to impress.",
    "I can't complain about the price."
]

# Define negation words
negation_words = {'not', 'no', 'never', "don't", "doesn't", "didn't", 
                  "won't", "can't", "couldn't", "hardly", "rarely"}

print("=== BIGRAMS FOR SENTIMENT CONTEXT ===")
print("Showing how bigrams capture negation that unigrams miss\n")

for review in reviews:
    tokens = word_tokenize(review.lower())
    bigrams = list(ngrams(tokens, 2))
    
    # Find negation bigrams
    neg_bigrams = [b for b in bigrams if b[0] in negation_words]
    
    # Overall sentiment
    sentiment = sia.polarity_scores(review)
    comp = sentiment['compound']
    label = "POSITIVE" if comp > 0.05 else ("NEGATIVE" if comp < -0.05 else "NEUTRAL")
    
    print(f"Review: \"{review}\"")
    if neg_bigrams:
        print(f"  Negation bigrams: {[' '.join(b) for b in neg_bigrams]}")
    print(f"  Sentiment: {label} (score: {comp:.2f})")
    print()

## 4. Visualizing a Bigram Network

In [ ]:
import networkx as nx

# Use filtered bigrams to create network
# Larger text sample for better network
sample_reviews = [
    "great camera quality amazing photos",
    "battery life excellent long lasting",
    "screen display bright colorful",
    "camera takes great photos low light",
    "battery drains fast charges slowly",
    "screen resolution high display quality",
    "camera zoom excellent quality photos",
    "battery life poor drains quickly",
    "great product excellent quality camera",
    "display bright screen beautiful colors"
]

# Get bigrams from all reviews
all_bigrams = []
for review in sample_reviews:
    words = review.lower().split()
    words = [w for w in words if w not in stop_words]
    all_bigrams.extend(list(ngrams(words, 2)))

bigram_freq = Counter(all_bigrams)

# Build network graph
G = nx.DiGraph()
for (w1, w2), count in bigram_freq.most_common(20):
    G.add_edge(w1, w2, weight=count)

# Visualize
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=2, seed=42)

# Node sizes based on degree
node_sizes = [300 + 200 * G.degree(node) for node in G.nodes()]

# Edge widths based on weight
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='#1565C0', alpha=0.8)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_color='white')
nx.draw_networkx_edges(G, pos, width=[w*1.5 for w in edge_weights], 
                        edge_color='#D32F2F', alpha=0.6, 
                        arrows=True, arrowsize=15)

plt.title('Bigram Network from Product Reviews', fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.savefig('bigram_network.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nHow to read this network:")
print("• Each circle = a word")
print("• Arrows = these words appear together (bigram direction)")
print("• Bigger circles = words that connect to more other words")
print("• Thicker arrows = more frequent bigrams")

## 5. Pairwise Word Correlation (Phi Coefficient)

In [ ]:
from itertools import combinations
import numpy as np
import seaborn as sns

# Sections (each review is a section)
sections = [
    "camera quality excellent battery life good",
    "screen bright display colorful camera good",
    "battery drains fast camera quality poor",
    "screen resolution high display excellent",
    "camera photos great battery charges fast",
    "display quality screen brightness excellent",
    "battery life poor phone overheats",
    "camera zoom excellent photos quality",
    "screen display touch responsive",
    "battery charging fast wireless excellent"
]

# Create word-section matrix
all_words = set()
section_words = []
for section in sections:
    words = set(section.lower().split()) - stop_words
    section_words.append(words)
    all_words.update(words)

# Filter to interesting words (appear in 2+ sections)
word_counts = Counter()
for words in section_words:
    word_counts.update(words)
focus_words = [w for w, c in word_counts.most_common(10) if c >= 2]

# Calculate Phi coefficient for each pair
def phi_coefficient(word1, word2, section_words):
    n = len(section_words)
    n11 = sum(1 for s in section_words if word1 in s and word2 in s)
    n10 = sum(1 for s in section_words if word1 in s and word2 not in s)
    n01 = sum(1 for s in section_words if word1 not in s and word2 in s)
    n00 = sum(1 for s in section_words if word1 not in s and word2 not in s)
    
    denom = np.sqrt((n11+n10)*(n11+n01)*(n00+n10)*(n00+n01))
    if denom == 0:
        return 0
    return (n11*n00 - n10*n01) / denom

print("=== PAIRWISE WORD CORRELATIONS (Phi Coefficient) ===")
print(f"{'Word Pair':<30} {'Phi':<10} {'Interpretation'}")
print("-" * 65)

correlations = []
for w1, w2 in combinations(focus_words, 2):
    phi = phi_coefficient(w1, w2, section_words)
    if abs(phi) > 0.1:  # Only show meaningful correlations
        interpretation = "Appear together" if phi > 0 else "Rarely together"
        correlations.append((f"{w1} & {w2}", phi, interpretation))

correlations.sort(key=lambda x: abs(x[1]), reverse=True)
for pair, phi, interp in correlations[:15]:
    print(f"{pair:<30} {phi:>+.3f}     {interp}")

# Correlation heatmap
n_words = len(focus_words)
corr_matrix = np.zeros((n_words, n_words))
for i, w1 in enumerate(focus_words):
    for j, w2 in enumerate(focus_words):
        if i == j:
            corr_matrix[i][j] = 1.0
        else:
            corr_matrix[i][j] = phi_coefficient(w1, w2, section_words)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, xticklabels=focus_words, yticklabels=focus_words,
            cmap='RdBu_r', center=0, annot=True, fmt='.2f', linewidths=0.5)
plt.title('Pairwise Word Correlation (Phi Coefficient)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('word_correlations.png', dpi=100)
plt.show()